# This is guidance for the pipelining side of things


## Building the pipeline

You can find the yml file in this folder, it's called **Transformations and Outputs.yaml**, to use this create a job as normal but then click the **Edit as YAML** option and copy and paste the code into the pipeline, it should auto-update when you save and navigate back to the ui view.

![image_1792076495234.png](./image_1792076495234.png "image_1792076495234.png")

### List of actions

1. Set up pipeline

2. Add in job parameters

3. Make sure personal computes are used

4. Add in databricks token sectret

5. Set over_a_week parameter in get_tables workbooks. The retention history is currently set to one 1 week, to find the code to set it for longer look in the "Changing Settings For Retention" notebook. To do this automatically you can combine the code in Changing Settings For Retention with the table names pulled out in the get_tables notebooks. 

6. Create volume in correct location outside of matt_test, and point get_tables, Run_QA and the dashboard data to it.


### Pipeline structure

You'll get a pipeline that looks like the following:

![image_1792076639521.png](./image_1792076639521.png "image_1792076639521.png")

It's broadly split into 5 parts:

1. Set up ![image_1792076671096.png](./image_1792076671096.png "image_1792076671096.png")

2. Running the code ![image_1792076693035.png](./image_1792076693035.png "image_1792076693035.png")

3. Generating the tables to run QA on ![image_1792076708404.png](./image_1792076708404.png "image_1792076708404.png")

4. Producing QA ![image_1792076726258.png](./image_1792076726258.png "image_1792076726258.png")

5. Building a dashboard from that QA ![image_1792076741460.png](./image_1792076741460.png "image_1792076741460.png")

### Pipeline settings

If you click a task you should see the following:

![image_1792076836427.png](./image_1792076836427.png "image_1792076836427.png")

**Key** things to take note of:

1. You'll need to **update** the **branch** to be the branch you're working on. Click edit and then type the correct branch in: ![image_1792076909549.png](./image_1792076909549.png "image_1792076909549.png")

2. Git provider is set so it runs from **Azure DevOps**, *not* **Databricks** so you will have to push to the main repo for changes to effect the pipeline.

3. We set up the Running of the QA to use R, this means that it can't use standard computes, either set up a **personal** or **team** compute with the most memory possible to run this step: ![image_1792077041372.png](./image_1792077041372.png "image_1792077041372.png")
![image_1792077066291.png](./image_1792077066291.png "image_1792077066291.png")

You'll be able to tell when this is done by the existing cluster id changing

![image_1792078089962.png](./image_1792078089962.png "image_1792078089962.png")

#### Jobs Parameters

There are also Job Parameters, these apply to every step of the pipeline, though really only need to be set for Set_Up_Parameters.ipynb.

You can find them on the right when clicking on the job.

![image_1792077178080.png](./image_1792077178080.png "image_1792077178080.png")

You'll need to make sure that the **DATABRICKS_SERVER_HOSTNAME** and **DATABRICKS_HTTP_PATH**
** are set up as normal and provide it with which level of term you need (1-term, 2-term, 3-term) so that it knows which folders to run through.

The year is somewhat unneccessary at the moment but have included it for future developement.


## Understanding the Pipeline



### Set_Up_Parameters

In this step we set up all the different parameters that are going to be required for the run.

We put them into taskValues, which can be created and accessed using set and get with a name and notebook (to go into the taskKey parameter).

In particular the tricky one is your Databricks pat token (this will be ingested by your parallel processes later to access the data in Databricks).

This is accessed in the following piece of code:


In [0]:
%python
import base64
from databricks.sdk import WorkspaceClient
workspaceclient = WorkspaceClient()
response = workspaceclient.secrets.get_secret(scope="pat_tokens", key="databricks_pat_token")

# SDK returns base64-encoded value — decode it
databricks_token = base64.b64decode(response.value).decode("utf-8")

dbutils.jobs.taskValues.set(key = "DATABRICKS_TOKEN", value = databricks_token)

To do this we have to use Databrick's method of secret keeping:



#### Secrets are done as follows


First you have to import WorkspaceClient

In [0]:
%python
from databricks.sdk import WorkspaceClient
workspaceclient = WorkspaceClient()


Then you create an internal folder or "scope" to store the secrets in.

In [0]:
%python
workspaceclient.secrets.create_scope(scope = "pat_tokens")

Then you use the following lines to put the secret the newly created folder,  
and replace `<your_pat_token>` with your own Databricks personal access token.
You can run this all here, it persists across and is a Databricks wide secret.

Scope is the folder, key the name of the secret and string_value the secret itself.

In [0]:
%python
workspaceclient.secrets.put_secret(  
    scope = "pat_tokens",
    key = "databricks_pat_token",
    string_value = "<your_pat_token>"
  )

Below is the code to get your secret back, with a given scope and key.

In [0]:
%python
import base64

response = workspaceclient.secrets.get_secret(scope="pat_tokens", key="databricks_pat_token")

It's written in base64 so you'll need to decode it.

In [0]:
decoded_resposne = base64.b64decode(response.value).decode("utf-8")


### Set_Up_Workspace_and_Pull_Out_SQL_Files

This simply sets up the workspace and pulls out the SQL files over which to run, if a transformation or output doesn't happen it's a good idea to check that the file generating it is in the list here, it may save you time waiting for the rest of the pipeline to run.

### Run_SQL_Transformations & Run_SQL_Outputs

Run_SQL_Tranformations & Run_SQL_Outputs are mirrored workbooks that do the same thing for the transformation SQL and output SQL files respectively.

These workbooks:

1. Set up parallel clusters

2. Split up the files across the cluster.

3. From each file pull out the SQL, making sure to split up statements, and remove comments in case of hidden.

4. Run each SQL statement in each file in turn.

**An important part of this is that each file for transformations and outputs are not dependent on any of the other files running previously.**

### QA_Transformations_get_tables & QA_Outputs_get_tables

Run_SQL_Tranformations & Run_SQL_Outputs are mirrored workbooks that do the same thing for the outputs of transformation SQL and output SQL files respectively.

These workbooks:

1. Set up parallel clusters

2. First pull out from the files a list of created tables in *get_transform_created_tables*.

3. Then pull out the current version of the table and previous version of the table in *get_transform_comparison_tables*.
    The previous version is defined as the youngest version that is older than the paramter *one_week_ago* or if that doesn't exist the oldest version younger than the parameter *one_week_ago*.
    We also have the parameter *timeframe_over_a_week* which denotes exactly which of these the previous version is.
    It is this step we need the token for, since each compute in the cluster is created without base access to Databricks.

4. *_make_serializable* then changes the type of the outputted objects from the parallel process to a form that can be saved as a json file, we do this because json is memory efficient and the files are two large to be passed between workbooks as taskValues.

We save each file to a different json file, indexed by a number and pass the number of files onto the QA workbooks in count.


### Run_QA_Transformations & Run_QA_Outputs
As before Run_QA_Tranformations & Run_QA_Outputs are mirrored workbooks that do the same thing for the outputs of transformation SQL and output SQL files respectively. & Run_QA_Outputs are mirrored workbooks that do the same thing for the outputs of QA_Tranformations_get_tables and QA_Tranformations_get_tables respectively.

These workbooks iterate over files, they read in the current and previous tables output a table containing a summary of whether the tables are the same or different.

Then we use versus to output different rows / columns, these are then saved in tables inside **matt_test**.

## Issues

### Running out of memory

We have to iterate over for loops as some of the files are large, remember to clean up memory when doing this.

### Runtime

It still takes a while, parallelisation (using Set_Up_Clusters.ipynb) was a means to speed this up.

## Enhancements

### Add a date to QA and don't recreate tables.

This would enable you to track how changes impact the outputs.

### Parallise the json write up step

Should be easy enough, just a copy of the above parallelisation.

### Parallise the QA

QA takes a time, might be worth implementing this: https://www.geeksforgeeks.org/r-language/parallel-programming-in-r/ then combine down, would involve a bit of a re-write.

### Add in years

Create a step at the start to run the most recent set-up step for the vw tables.

### Continue Jay's set-up

Eventually transition from using the term parameter to select which files, to altering a single file to run differently based off term.

Would start with:

1. Giant if statement containing everything.

2. Parse down the repeated segments only changing the parts needed.